# Cortex Management Plane Exploration

Notebook exploratorio para management audit logs del Cortex Management Plane con foco en:

- validacion incremental del endpoint oficial
- exploracion real del esquema
- derivacion de features operativas
- primeras detecciones de anomalias administrativas
- evaluacion de Sentinel como base reusable para este caso de uso

El endpoint objetivo es el oficial de Cortex XDR para management audit logs:

- `POST /public_api/v1/audits/management_logs`

Este notebook esta pensado para una primera iteracion. Prioriza trazabilidad, claridad y capacidad de crecer a una ventana historica mayor.

In [1]:
import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from IPython.display import display

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / 'pyproject.toml').exists():
    for candidate in [REPO_ROOT, *REPO_ROOT.parents]:
        if (candidate / 'pyproject.toml').exists():
            REPO_ROOT = candidate
            break

sys.path.insert(0, str(REPO_ROOT / 'src'))
sys.path.insert(0, str(REPO_ROOT / 'usecases' / 'Cortex' / 'src'))

from sentinel.explorer import Thresholds
from sentinel.visualization import AnomalyVisualizer

from cortex_usecase import (
    CortexClientConfig,
    CortexManagementAuditClient,
    CortexManagementAuditParser,
    build_anomaly_taxonomy_table,
    build_field_interpretation_table,
    build_schema_profile,
    build_window_features,
    dataframe_to_csv,
    json_dump,
    link_events_to_anomalous_windows,
    make_extraction_id,
    normalize_management_audit_dataframe,
    records_to_dataframe,
    resolve_time_window,
    resolve_usecase_paths,
    run_isolation_forest_detection,
    run_rrcf_detection,
    run_signal_review,
)

pd.set_option('display.max_columns', 120)
pd.set_option('display.max_rows', 200)
pd.set_option('display.max_colwidth', 120)
plt.style.use('seaborn-v0_8-whitegrid')

ModuleNotFoundError: No module named 'matplotlib'

## 1. Introduccion

### Por que management plane

Los audit logs del management plane concentran actividad administrativa sensible: cambios de politica, gestion de agentes, tokenizacion, permisos, Action Center y otras operaciones que pueden alterar la cobertura, la estabilidad o la disponibilidad del servicio.

### Por que audit logs

Porque no buscamos solo telemetria tecnica del endpoint. Buscamos cambios administrativos que anticipen degradacion operativa:

- desinstalacion masiva de agentes y perdida de cobertura
- generacion anomala de tokens o API keys
- bursts de cambios administrativos fuera del patron habitual
- actividad inusual por usuario, IP o user-agent
- aumentos de fallos operativos o de autenticacion
- secuencias de acciones que elevan el riesgo de impacto operativo

### Alcance de esta iteracion

Esta primera iteracion no intenta afinar un modelo final. Intenta dejar un pipeline serio para:

- extraer una ventana corta
- validar autenticacion y acceso
- observar el esquema real
- persistir artefactos reproducibles
- crear features iniciales
- correr una primera deteccion con Sentinel

## 2. Estrategia de validacion incremental

Antes de ampliar el rango temporal, este notebook arranca con una ventana pequena por defecto. La idea es deliberada:

- reducir volumen y tiempos de respuesta
- validar que el endpoint y la autenticacion responden correctamente
- verificar parsing y persistencia de artefactos
- inspeccionar el esquema real antes de asumir campos analiticos
- validar que Sentinel se integra bien con el dataset antes de escalar

Un resultado importante de esta fase puede ser "la ventana aun no da suficiente historico para un baseline fuerte". Eso no invalida el ejercicio; al contrario, ayuda a decidir el siguiente incremento.

## 3. Configuracion

Se dejan configurables los parametros pedidos para poder mantener una validacion incremental y luego crecer el alcance con cambios minimos.

In [ ]:
paths = resolve_usecase_paths(REPO_ROOT)

FAST_TEST_MODE = True
LOOKBACK_HOURS = 6
START_TIME = None
END_TIME = None
INITIAL_OFFSET = 0
PAGE_SIZE = 100
MAX_RECORDS = 100 if FAST_TEST_MODE else 5000
TIME_WINDOW = '15min'
ROLLING_WINDOW_SIZE = 4
LOCAL_TIMEZONE = 'America/Bogota'
USE_SAMPLE_IF_NO_CREDS = True

start_dt, end_dt = resolve_time_window(
    lookback_hours=LOOKBACK_HOURS,
    start_time=START_TIME,
    end_time=END_TIME,
)
extraction_id = make_extraction_id(start_dt, end_dt)

sample_fixture_path = paths.raw_dir / 'sample_management_audit_logs.json'
raw_output_path = paths.raw_dir / f'{extraction_id}.json'
metadata_output_path = paths.processed_dir / f'{extraction_id}_metadata.json'
processed_events_path = paths.processed_dir / f'{extraction_id}_events.csv'
processed_features_path = paths.processed_dir / f'{extraction_id}_window_features.csv'
field_table_path = paths.processed_dir / f'{extraction_id}_field_interpretation.csv'
taxonomy_table_path = paths.processed_dir / f'{extraction_id}_anomaly_taxonomy.csv'

config_snapshot = {
    'FAST_TEST_MODE': FAST_TEST_MODE,
    'LOOKBACK_HOURS': LOOKBACK_HOURS,
    'START_TIME': START_TIME,
    'END_TIME': END_TIME,
    'INITIAL_OFFSET': INITIAL_OFFSET,
    'PAGE_SIZE': PAGE_SIZE,
    'MAX_RECORDS': MAX_RECORDS,
    'TIME_WINDOW': TIME_WINDOW,
    'ROLLING_WINDOW_SIZE': ROLLING_WINDOW_SIZE,
    'LOCAL_TIMEZONE': LOCAL_TIMEZONE,
    'USE_SAMPLE_IF_NO_CREDS': USE_SAMPLE_IF_NO_CREDS,
    'resolved_start_time': start_dt.isoformat(),
    'resolved_end_time': end_dt.isoformat(),
}

pd.DataFrame([config_snapshot]).T.rename(columns={0: 'value'})

## 4. Cliente API

La implementacion reusable del cliente hace cinco cosas:

- genera headers autenticados para `advanced` y `standard`
- valida autenticacion usando el mismo endpoint objetivo con una consulta minima
- pagina resultados con `search_from` / `search_to`
- persiste raw JSON y metadatos de extraccion
- transforma la respuesta en `DataFrame` sin asumir un parser fijo del repo

Importante: no se hardcodean secretos. Todo viene de variables de entorno.

In [ ]:
required_env = ['XDR_BASE_URL', 'XDR_API_KEY_ID', 'XDR_API_KEY']
has_credentials = all(os.getenv(name) for name in required_env)

client_config = CortexClientConfig.from_env() if has_credentials else None
runtime_mode = 'live_api' if has_credentials else 'sample_fixture'

pd.DataFrame(
    [
        {
            'runtime_mode': runtime_mode,
            'credentials_present': has_credentials,
            'base_url': getattr(client_config, 'base_url', None),
            'auth_mode': getattr(client_config, 'auth_mode', None),
            'page_size': getattr(client_config, 'page_size', None),
        }
    ]
)

## 5. Ingesta y persistencia

La extraccion se hace de forma incremental. Si hay credenciales, el notebook intenta validar autenticacion y luego extraer una ventana corta. Si no hay credenciales, usa un fixture sintetico para probar el pipeline de punta a punta sin afirmar que ese fixture represente el esquema real del tenant.

In [ ]:
validation_summary = None
raw_payload = None
persisted_raw_path = raw_output_path

if has_credentials:
    client = CortexManagementAuditClient(client_config)
    validation_summary = client.validate_authentication(start_time=start_dt, end_time=end_dt)
    raw_payload = client.fetch_management_audit_logs(
        start_time=start_dt,
        end_time=end_dt,
        initial_offset=INITIAL_OFFSET,
        page_size=PAGE_SIZE,
        max_records=MAX_RECORDS,
    )
    json_dump(raw_payload, persisted_raw_path)
    raw_df = CortexManagementAuditParser(str(persisted_raw_path)).parse()
else:
    if not USE_SAMPLE_IF_NO_CREDS:
        raise RuntimeError('No credentials found and USE_SAMPLE_IF_NO_CREDS is False.')
    persisted_raw_path = sample_fixture_path
    raw_payload = json.loads(sample_fixture_path.read_text(encoding='utf-8'))
    raw_df = CortexManagementAuditParser(str(sample_fixture_path)).parse()

extraction_metadata = {
    'runtime_mode': runtime_mode,
    'validation_summary': validation_summary,
    'persisted_raw_path': str(persisted_raw_path),
    'metadata_output_path': str(metadata_output_path),
    'processed_events_path': str(processed_events_path),
    'processed_features_path': str(processed_features_path),
    'record_count': int(len(raw_df)),
    'raw_columns': list(raw_df.columns),
    'start_time': start_dt.isoformat(),
    'end_time': end_dt.isoformat(),
    'note': 'If runtime_mode == sample_fixture, schema conclusions remain provisional until a live extraction is executed.',
}
json_dump(extraction_metadata, metadata_output_path)

display(pd.DataFrame([extraction_metadata]).T.rename(columns={0: 'value'}))
raw_df.head(10)

## 6. Exploracion del esquema

En esta seccion no asumimos que los campos documentados sean exactamente los que devolveran todos los tenants o todas las versiones. El objetivo es construir una vista operativa del esquema realmente observado:

- columnas encontradas
- tipos
- nulos
- cardinalidad
- ejemplos
- estabilidad esperada
- utilidad analitica potencial

In [ ]:
schema_profile = build_schema_profile(raw_df)
display(schema_profile)

print('Shape:', raw_df.shape)
print('Columnas observadas:', list(raw_df.columns))
print('Rango temporal bruto:')
if 'AUDIT_INSERT_TIME' in raw_df.columns:
    audit_ts = pd.to_datetime(pd.to_numeric(raw_df['AUDIT_INSERT_TIME'], errors='coerce'), unit='ms', utc=True, errors='coerce')
    print(audit_ts.min(), '->', audit_ts.max())

## 7. Interpretacion de campos

La tabla siguiente aterriza una primera interpretacion de los campos observados. No pretende reemplazar documentacion oficial del proveedor. Sirve para decidir rapido que columnas merecen mas atencion analitica y cuales conviene tratar como meramente auxiliares.

In [ ]:
field_table = build_field_interpretation_table(raw_df)
dataframe_to_csv(field_table, field_table_path)
display(field_table)

## 8. Taxonomia de anomalias del caso de uso

Antes de modelar, conviene separar las familias de anomalias que nos interesan. Algunas son fuertes incluso con poco historico; otras requieren memoria de comportamiento por usuario, IP o sesion.

In [ ]:
normalized_df = normalize_management_audit_dataframe(raw_df, local_timezone=LOCAL_TIMEZONE)
dataframe_to_csv(normalized_df, processed_events_path)

preview_columns = [
    'event_time_local', 'owner_identity', 'entity', 'entity_subtype', 'result_normalized',
    'source_ip', 'user_agent', 'mentions_uninstall', 'mentions_token', 'mentions_api_key',
    'mentions_config', 'mentions_action_center', 'is_failure', 'is_high_risk_sequence'
]
preview_columns = [column for column in preview_columns if column in normalized_df.columns]
normalized_df[preview_columns].head(15)

## 9. Ingenieria de variables

La capa de features agrega eventos administrativos en ventanas cortas. El objetivo no es solo contar eventos, sino preservar senales con valor operativo:

- conteos por ventana
- admins unicos
- IPs y user-agents unicos
- ratio de fallos
- actividad fuera de horario
- first-seen IP / user-agent por usuario
- rare actions
- flags semanticos para uninstall, token, API key, config y Action Center
- secuencias administrativas de alto riesgo

Ademas, se agregan rolling means y desviaciones para ayudar a Sentinel a capturar bursts relativos y no solo volumen absoluto.

In [ ]:
feature_df = build_window_features(
    normalized_df,
    time_window=TIME_WINDOW,
    rolling_window_size=ROLLING_WINDOW_SIZE,
)
dataframe_to_csv(feature_df.reset_index(), processed_features_path)

display(feature_df.head(20))
print('Feature shape:', feature_df.shape)
print('Feature columns:', list(feature_df.columns))

## 10. Integracion con Sentinel

### 10.1 Preparacion / compatibilidad

No existe un parser nativo de Cortex en Sentinel. Para esta iteracion se resolvio una integracion compatible con el estilo del repo:

- `CortexManagementAuditParser` extiende `BaseLogParser`
- `StringAggregator` crea agregados por ventana sobre columnas categoricas y semanticas
- `RollingAggregator` enriquece la serie agregada con contexto temporal

### 10.2 Explorer

Se usa `SignalDiagnostics` para estimar si la ventana actual tiene suficiente senal. Con ventanas muy cortas es esperable que falle por volumen historico, y ese resultado tambien es util.

### 10.3 Aggregators

Se reutilizan `StringAggregator` y `RollingAggregator` de Sentinel como parte real del pipeline, no solo como ejemplo decorativo.

### 10.4 Detection

Se ejecuta `IsolationForestDetector` como detector base. `RRCFDetector` queda habilitado de forma opcional cuando la dependencia esta instalada.

In [ ]:
signal_review = run_signal_review(feature_df, thresholds=Thresholds.relaxed())
signal_summary = signal_review['summary'].sort_values(['iqr_anomaly_pct', 'variance'], ascending=False)

print(signal_review['quality_report'])
print()
print(signal_review['interpretation'])
display(signal_summary.head(15))

In [ ]:
iforest_result = run_isolation_forest_detection(
    feature_df,
    contamination=0.10 if FAST_TEST_MODE else 0.05,
)
detected_df = iforest_result['detected']
iforest_threshold = iforest_result['threshold']

rrcf_result = run_rrcf_detection(feature_df)
anomalous_windows = detected_df.loc[detected_df['iforest_label'] == -1].sort_values('iforest_score', ascending=False)

display(
    anomalous_windows[
        [
            'iforest_score', 'events_count', 'failure_count', 'high_risk_action_count',
            'uninstall_flag_count', 'token_flag_count', 'api_key_flag_count',
            'high_risk_sequence_count', 'rare_action_count'
        ]
    ].head(10)
)

if rrcf_result is None:
    print('RRCF no se ejecuto. Instala el extra [rrcf] si quieres comparar detectores.')
else:
    display(rrcf_result['detected'][['rrcf_score', 'rrcf_label', 'events_count']].sort_values('rrcf_score', ascending=False).head(10))

## 11. Visualizacion

Se prioriza el stack de Sentinel cuando es razonable. Aqui se combinan dos enfoques:

- graficos propios con `matplotlib` para volumetria, top categorias y fallos
- `AnomalyVisualizer` de Sentinel para score timeline, distribucion y overlays de features

In [ ]:
overview_path = paths.figures_dir / f'{extraction_id}_overview.png'

top_entities = normalized_df['entity'].value_counts().head(8)
top_users = normalized_df['owner_identity'].value_counts().head(8)
top_ips = normalized_df['source_ip'].dropna().astype(str).value_counts().head(8) if normalized_df['source_ip'].notna().any() else pd.Series(dtype=int)

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

detected_df[['events_count', 'failure_count', 'high_risk_action_count']].plot(ax=axes[0, 0], linewidth=1.6)
axes[0, 0].set_title('Actividad administrativa por ventana')
axes[0, 0].set_xlabel('Window start')
axes[0, 0].set_ylabel('Count')

top_entities.sort_values().plot(kind='barh', ax=axes[0, 1], color='#1f77b4')
axes[0, 1].set_title('Top tipos / entidades')
axes[0, 1].set_xlabel('Eventos')

top_users.sort_values().plot(kind='barh', ax=axes[1, 0], color='#ff7f0e')
axes[1, 0].set_title('Actividad por usuario')
axes[1, 0].set_xlabel('Eventos')

axes[1, 1].plot(detected_df.index, detected_df['iforest_score'], color='#444444', linewidth=1.5, label='IForest score')
axes[1, 1].scatter(
    detected_df.index[detected_df['iforest_label'] == -1],
    detected_df.loc[detected_df['iforest_label'] == -1, 'iforest_score'],
    color='crimson',
    label='Anomalia',
    zorder=5,
)
axes[1, 1].axhline(iforest_threshold, color='darkorange', linestyle='--', linewidth=1.2, label='Threshold')
axes[1, 1].set_title('Scores de anomalia por ventana')
axes[1, 1].set_xlabel('Window start')
axes[1, 1].set_ylabel('Score')
axes[1, 1].legend(loc='best')

fig.tight_layout()
fig.savefig(overview_path, dpi=150, bbox_inches='tight')
display(overview_path)

if not top_ips.empty:
    display(top_ips.rename_axis('source_ip').reset_index(name='event_count'))

In [ ]:
viz_df = detected_df.copy()
viz = AnomalyVisualizer(viz_df, score_col='iforest_score', anomaly_col='iforest_label')
viz.plot_static(
    title='Isolation Forest sobre ventanas de audit logs administrativos',
    threshold=iforest_threshold,
    ylabel='Score (alto = mas anomalo)',
)
viz.plot_score_distribution(
    threshold=iforest_threshold,
    title='Distribucion de scores: ventanas normales vs anomalias',
)

feature_overlay_columns = [
    column for column in [
        'events_count', 'failure_count', 'high_risk_action_count',
        'uninstall_flag_count', 'token_flag_count', 'api_key_flag_count'
    ]
    if column in viz_df.columns
]
viz.plot_features(
    feature_columns=feature_overlay_columns,
    title='Features clave con ventanas anomalias resaltadas',
)

anomalous_event_df = link_events_to_anomalous_windows(
    normalized_df,
    viz_df,
    time_window=TIME_WINDOW,
    label_column='iforest_label',
    score_column='iforest_score',
)
event_view_columns = [
    'event_time_local', 'owner_identity', 'entity', 'entity_subtype', 'result_normalized',
    'source_ip', 'user_agent', 'iforest_score', 'description'
]
event_view_columns = [column for column in event_view_columns if column in anomalous_event_df.columns]
anomalous_event_df[event_view_columns].head(25)

## 12. Hallazgos

Esta seccion no pretende cerrar el caso de uso. Busca dejar una lectura tecnica clara de lo que ya muestra valor y de lo que todavia depende de mas historico.

In [ ]:
candidate_windows = detected_df.sort_values('iforest_score', ascending=False).head(5)
candidate_window_columns = [
    column for column in [
        'iforest_score', 'events_count', 'failure_count', 'high_risk_action_count',
        'uninstall_flag_count', 'token_flag_count', 'api_key_flag_count',
        'high_risk_sequence_count', 'rare_action_count', 'outside_business_hours_count'
    ]
    if column in candidate_windows.columns
]

print('Campos mas prometedores observados en esta corrida:')
display(field_table[field_table['relevancia_analitica'] == 'Alta'].head(12))

print('Features con mas outliers IQR segun Explorer:')
display(signal_summary[['feature', 'variance', 'iqr_anomaly_pct']].head(12))

print('Ventanas candidatas a anomalia segun Isolation Forest:')
display(candidate_windows[candidate_window_columns])

if not anomalous_event_df.empty:
    print('Eventos contenidos en ventanas anomalias:')
    display(anomalous_event_df[event_view_columns].head(25))

## Tablas Finales Obligatorias

Las dos tablas siguientes quedan listas para exportarse y reutilizarse en la siguiente iteracion.

In [ ]:
taxonomy_table = build_anomaly_taxonomy_table(normalized_df, feature_df)
dataframe_to_csv(taxonomy_table, taxonomy_table_path)

print('Tabla 1: interpretacion de campos')
display(field_table)

print('Tabla 2: taxonomia de anomalias')
display(taxonomy_table)

## 13. Conclusiones y siguientes pasos

### Viabilidad del caso de uso

Sentinel si es una base util para este escenario. No porque traiga un parser nativo de Cortex, sino porque ya resuelve piezas importantes del pipeline:

- ingestion style reusable mediante `BaseLogParser`
- agregacion por ventana con `StringAggregator`
- contexto temporal adicional con `RollingAggregator`
- validacion de senal con `SignalDiagnostics`
- primera deteccion no supervisada con `IsolationForestDetector`
- visualizacion simple de scores y features con `AnomalyVisualizer`

### Limites de esta iteracion

- si la corrida es muy corta, el baseline por usuario, IP y user-agent sigue siendo debil
- cualquier conclusion sobre esquema sigue siendo provisional hasta ejecutar una extraccion real en tu tenant
- para secuencias y habitualidad operativa conviene ampliar al menos a varios dias

### Siguiente iteracion recomendada

1. ejecutar una corrida real de 6 a 24 horas y revisar esquema observado
2. ampliar a 7 a 14 dias para habitualidad por usuario, IP y user-agent
3. consolidar una taxonomia mas precisa de acciones criticas por `entity` y `entity_subtype`
4. incorporar correlacion con cobertura de agentes, errores de integracion y cambios de politica
5. comparar Isolation Forest con RRCF cuando el historico y la dependencia opcional esten disponibles